In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Every ingredient of the Karplus-Strong algorithm is on a control here. The
plot is a spectrogram of the note, up to 5 kHz, with each harmonic as a
horizontal line that fades as it decays. Switch the low-pass filter off and
every harmonic fades at the same rate. Switch the excitation to a unit
impulse and the attack loses its noise. Push the pitch up and watch the
readout: rounding $M$ to a whole number of samples throws the high notes out
of tune.

In [ ]:
# hide
# autorun
PITCH0, T0, LOWPASS0, EXCITE0 = "G3", 2.5, True, "White noise"

SR = 44100
T_PLAY = 6.0                        # every note is rendered for 6 s
N = int(T_PLAY * SR)
NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
PITCHES = [f"{NAMES[m % 12]}{m // 12 - 1}" for m in range(40, 89)]   # E2 to E6
FREQ = {name: 440 * 2 ** ((m - 69) / 12) for name, m in zip(PITCHES, range(40, 89))}
NOISE = np.random.default_rng(1).uniform(-1.0, 1.0, SR)    # one fixed burst

N_FFT, HOP, F_TOP = 2048, 2048, 5000.0
WIN = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT))
FB = np.fft.rfftfreq(N_FFT, 1 / SR)
KEEP = FB <= F_TOP
TF = (np.arange((N - N_FFT) // HOP + 1) * HOP + N_FFT / 2) / SR


def karplus_strong(f, T, lowpass, noise, SR=SR, N=N, NOISE=NOISE):
    """y[n] = x[n] + g * (y[n - M] + y[n - M - 1]) / 2, computed one period
    (M samples) at a time: each period depends only on the one before."""
    M = round(SR / f)
    g = 10 ** (-3 * M / (SR * T))
    x = np.zeros(N)
    if noise:
        x[:M] = NOISE[:M]
    else:
        x[0] = 1.0
    z = np.zeros(N + M + 1)         # z[n + M + 1] = y[n], with zeros before n = 0
    for a in range(0, N, M):
        b = min(a + M, N)
        if lowpass:
            fb = 0.5 * (z[a + 1:b + 1] + z[a:b])   # y[n - M] and y[n - M - 1]
        else:
            fb = z[a + 1:b + 1]
        z[a + M + 1:b + M + 1] = x[a:b] + g * fb
    return z[M + 1:], M, g


FRAME_IDX = np.arange(0, N - N_FFT + 1, HOP)[:, None] + np.arange(N_FFT)


def spectrogram(y, WIN=WIN, FRAME_IDX=FRAME_IDX, KEEP=KEEP):
    frames = y[FRAME_IDX]           # one row per frame
    S = np.abs(np.fft.rfft(frames * WIN, axis=1))[:, KEEP]
    db = 20 * np.log10(np.maximum(S / max(S.max(), 1e-12), 1e-5))
    return np.round(np.maximum(db, -80.0)).astype(np.int8).T   # whole dB keep it light


def figure():
    y = karplus_strong(FREQ[PITCH0], T0, LOWPASS0, EXCITE0 == "White noise")[0]
    fig = go.Figure()
    fig.add_heatmap(x=TF, y=FB[KEEP], z=spectrogram(y), zmin=-80, zmax=0,
                    colorscale=[[0, "#ffffff"], [0.45, "#9ec5e0"],
                                [0.8, BLUE], [1, "#002a45"]],
                    showscale=False, hoverinfo="skip")
    fig.update_xaxes(range=[0, T_PLAY], title_text="Time (s)", fixedrange=True)
    fig.update_yaxes(range=[0, F_TOP], title_text="Frequency (Hz)",
                     fixedrange=True)
    return fig


def controls(fig):
    pitch = widgets.SelectionSlider(description="Pitch", options=PITCHES,
                                    value=PITCH0)
    T = widgets.FloatSlider(description="Decay $T$ (s)", min=0.25, max=6.0,
                            step=0.05, value=T0)
    lowpass = widgets.Checkbox(description="Low-pass filter in the loop",
                               value=LOWPASS0)
    excite = widgets.Dropdown(description="Excitation", value=EXCITE0,
                              options=["White noise", "Unit impulse"])
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(pitch, T, lowpass, excite, karplus_strong=karplus_strong,
               spectrogram=spectrogram, FREQ=FREQ, SR=SR, readout=readout):
        f = FREQ[pitch]
        y, M, g = karplus_strong(f, T, lowpass, excite == "White noise")
        with fig.batch_update():
            fig.data[0].z = spectrogram(y)
        cents = 1200 * np.log2(SR / M / f)
        readout.value = (
            f"<span style='font-size:0.9em'><i>M</i> = round(<i>f</i><sub>s</sub>"
            f" / <i>f</i>) = round({SR:,} / {f:.1f}) = {M} samples &rarr; "
            f"{SR / M:.1f} Hz ("
            f"{f'{cents:+.0f}'.replace('-', '&minus;')} cents) &nbsp;·&nbsp; <i>g</i> = 10<sup>&minus;3<i>M</i> / "
            f"(<i>f</i><sub>s</sub> <i>T</i>)</sup> = {g:.4f}</span>")

    widgets.interactive_output(update, {"pitch": pitch, "T": T,
                                        "lowpass": lowpass, "excite": excite})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(karplus_strong=karplus_strong, FREQ=FREQ):
        y = karplus_strong(FREQ[pitch.value], T.value, lowpass.value,
                           excite.value == "White noise")[0]
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (pitch, T, lowpass, excite):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([pitch, T, lowpass, excite, readout, out, gate])

icm_plotly.show(figure, controls)